# Credit Card Fraud Detection — Pattern Analysis
### Can we identify fraud without machine learning?

---

**Dataset:** 284,807 real credit card transactions | 492 confirmed fraud cases  
**Tools:** Python | MySQL | Pandas | Matplotlib | Scipy  
**Method:** Pure statistical analysis + rule-based flag system  
**Source:** Kaggle — ULB Machine Learning Group  

---

This project investigates whether meaningful fraud patterns exist 
in transaction data that can be identified through SQL analysis 
and statistics alone — without building a machine learning model.

The dataset is severely imbalanced — only 0.17% of transactions 
are fraud. The core challenge is finding signals strong enough to 
rise above that noise.

**Seven questions were investigated:**

1. How imbalanced is the data and why does it matter?
2. Does fraud cluster at specific hours of the day?
3. Are fraudulent transaction amounts different from legitimate ones?
4. Which amount ranges carry the highest fraud risk?
5. How does fraud volume change day on day?
6. Do repeated transactions in the same minute signal fraud?
7. Which dataset features correlate most strongly with fraud?

All findings feed into a final rule-based flag system that assigns 
every transaction a risk score of 0 to 3 — built entirely from 
patterns discovered in the analysis.

In [4]:
import pandas as pd
import mysql.connector
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats


conn = mysql.connector.connect(
    host     = '127.0.0.1',
    port     = 3306,
    user     = 'root',
    password = 'YOUR_PASSWORD',
    database = 'fraud_db'
)


def run_query(query):
    cursor = conn.cursor()
    cursor.execute(query)
    rows    = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    return pd.DataFrame(rows, columns=columns)


df = pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\data\creditcard.csv')

print("Connected to MySQL ✓")
print("Dataset loaded:", df.shape, "✓")

ProgrammingError: 1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)

In [5]:
import os
import mysql.connector

password = os.environ.get('MYSQL_PASSWORD')

if password is None:
    print("⚠️ MYSQL_PASSWORD not found in environment — the setx won't apply until you restart Jupyter/terminal")
else:
    try:
        conn = mysql.connector.connect(
            host='127.0.0.1',
            port=3306,
            user='root',
            password=password
        )
        print("Connected ✓")
        cursor = conn.cursor()
        cursor.execute("SHOW DATABASES")
        for db in cursor:
            print(db)
        conn.close()
    except mysql.connector.Error as e:
        print("ERROR CODE:", e.errno)
        print("ERROR MSG:", e.msg)

⚠️ MYSQL_PASSWORD not found in environment — the setx won't apply until you restart Jupyter/terminal


 ## Finding 1 — Class Imbalance This section explores how rare fraud is in the dataset and why this matters for detection.

In [ ]:

df1=pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\sql_results\data_imbalance.csv')
fig,ax=plt.subplots(figsize=(8,5))
colors = ['#2E75B6', '#C0392B']
bars=ax.bar(['Legitimate','Fraud'],df1['TOTAL'], color=colors, width=0.5)
for bar, pct in zip(bars, df1['percentage']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1000,
            f'{pct}%', ha='center', fontweight='bold', fontsize=12)
ax.set_title('Transaction Class Distribution\n(0.17% of transactions are fraud)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Transactions')
ax.set_xlabel('Transaction Class')
plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

Out of 284,807 transactions, only 492 are fraud  just 0.17%. This severe imbalance means accuracy is a completely misleading metric. A model that predicts "legitimate" for every single transaction would be 99.83% accurate but catch zero fraud. Every finding in this project is evaluated against this 0.17% baseline any pattern that exceeds this rate is a genuine signal worth acting on.

 ## Finding 2 — Does fraud cluster at specific hours? 

In [ ]:

df2=pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\sql_results\frauds_by_hour.csv')
df2 = df2.sort_values('hour_num')


peak_hours = df2.nlargest(4, 'fraud_rate')['hour_num'].tolist()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df2['hour_num'], df2['fraud_rate'],
        color='#C0392B', linewidth=2.5, marker='o', markersize=5)


for h in peak_hours:
    ax.axvspan(h-0.5, h+0.5, alpha=0.2, color='red', label='_nolegend_')

ax.set_title('Fraud Rate by Hour of Day\n(Shaded = peak fraud hours)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day  (0 = Midnight, 12 = Noon, 23 = 11PM)')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\hourlyFraud.png', dpi=150, bbox_inches='tight')
plt.show()


Hour 2 dominates by both metrics most fraud transactions (57) and highest fraud rate (1.71%). Hour 4 reveals a hidden pattern —ranked only #6 by volume but #2 by rate at 1.04%, indicating concentrated fraud during low-traffic periods where fewer legitimate transactions provide cover.

 ## Finding 3 - Is the average fraud amount higher or lower than legitimate? Does the shape of the distribution differ between the two classes?

In [ ]:
df = pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\data\creditcard.csv')
fraud = df[df['Class'] == 1]['Amount']
legit = df[df['Class'] == 0]['Amount']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


axes[0].hist(legit[legit < 500], bins=60,
             color='#2E75B6', alpha=0.8, edgecolor='white')
axes[0].axvline(legit.mean(), color='navy', linestyle='--', linewidth=2,
                label=f'Mean: ${legit.mean():.2f}')
axes[0].set_title('Legitimate Transactions', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')
axes[0].legend()


axes[1].hist(fraud[fraud < 500], bins=60,
             color='#C0392B', alpha=0.8, edgecolor='white')
axes[1].axvline(fraud.mean(), color='darkred', linestyle='--', linewidth=2,
                label=f'Mean: ${fraud.mean():.2f}')
axes[1].set_title('Fraudulent Transactions', fontweight='bold')
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Amount Distribution: Fraud vs Legitimate',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\amtDist.png', dpi=150, bbox_inches='tight')
plt.show()


Legitimate transactions have a mean of $88.29 while fraudulent transactions have a higher mean of $122.21. However both distributions are heavily skewed — the vast majority of both fraud and legitimate transactions cluster near zero. The fraud distribution shows a distinct secondary spike around $100-$130, suggesting a subset of fraud involves medium-sized amounts. The higher fraud mean is driven by outliers, not the typical fraud transaction.

 ## Finding 4 - Which amount bucket has the highest fraud rate? Does the pattern suggest one fraud strategy or multiple? 

In [ ]:
df4 = pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\sql_results\amt_bucket.csv')


fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2E75B6' if r < max(df4['FRAUD_RATE']) 
          else '#C0392B' for r in df4['FRAUD_RATE']]

bars = ax.bar(df4['AMOUNT_CATEGORY'], df4['FRAUD_RATE'], 
              color=colors, width=0.5)


for bar, val in zip(bars, df4['FRAUD_RATE']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{val}%', ha='center', fontweight='bold', fontsize=11)


ax.axhline(y=0.17, color='black', linestyle='--', 
           linewidth=1.5, label='Baseline (0.17%)')

ax.set_title('Fraud Rate by Transaction Amount Bucket\n(Dashed line = dataset baseline)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('Amount Category')
ax.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\amtBucket.png', dpi=150, bbox_inches='tight')
plt.show()

All four amount buckets exceed the 0.17% baseline except Small ($10-$100) at just 0.087%  the safest bucket. Large transactions ($1000+) have the highest fraud rate at 0.293%, nearly double the baseline, suggesting high-value card abuse. Micro transactions (under $10) are the second most suspicious at 0.256%, consistent with card testing behaviour where fraudsters make tiny transactions to verify stolen card details before larger purchases. Two completely different fraud strategies are visible in this one chart.

## Finding 5- How does fraud volume change from one day to the next? 

In [ ]:
df5 = pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\sql_results\day_to_day.csv')


fig, ax = plt.subplots(figsize=(8, 5))


bars = ax.bar(df5['day_num'], df5['frauds'],
              color=['#C0392B', '#E8A0A0'], width=0.4)


for bar, val in zip(bars, df5['frauds']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1,
            str(int(val)), ha='center',
            fontweight='bold', fontsize=13)


for _, row in df5.iterrows():
    if pd.notna(row['fraud_change']):
        change = int(row['fraud_change'])
        symbol = '▲' if change > 0 else '▼'
        color  = '#C0392B' if change > 0 else '#1A5C38'
        ax.text(row['day_num'], row['frauds'] / 2,
                f'{symbol} {abs(change)} vs yesterday',
                ha='center', fontsize=11,
                color='white', fontweight='bold')

ax.set_title('Day on Day Fraud Count\n(Change shown inside bar)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Fraud Transactions')
ax.set_xlabel('Day')
ax.set_xticks(df5['day_num'])
ax.set_xticklabels([f'Day {int(d)}' for d in df5['day_num']])




plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\daily_trend.png',
            dpi=150, bbox_inches='tight')
plt.show()

Day 0 recorded 281 fraud transactions while Day 1 dropped to 211  a decrease of 70 cases (24.9% reduction). While the dataset covers only 48 hours which limits trend conclusions, the LAG() function used here is the foundation of real production fraud monitoring systems. On a dataset spanning months, this same query would immediately surface any day where fraud spikes abnormally triggering an alert for the fraud operations team.

 ## Finding 6 - Are there multiple transactions for the same amount within the same minute? 

In [ ]:
df6 = pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\sql_results\dup_transactions.csv')


df6_top = df6.sort_values(['fraudPerGroup', 'numOfTransactions'], 
                           ascending=False).head(15)


colors = ['#C0392B' if f > 0 else '#2E75B6' for f in df6_top['fraudPerGroup']]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(range(len(df6_top)), df6_top['numOfTransactions'],
              color=colors, width=0.6)


ax.set_xticks(range(len(df6_top)))
ax.set_xticklabels([f'${v}' for v in df6_top['AMT']], 
                    rotation=45, ha='right', fontsize=10)


from matplotlib.patches import Patch
legend = [Patch(color='#C0392B', label='Contains Fraud'),
          Patch(color='#2E75B6', label='No Fraud')]
ax.legend(handles=legend)

ax.set_title('Top 15 Duplicate Transaction Groups\n(Red = contains fraud, Blue = no fraud)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Duplicate Transactions')
ax.set_xlabel('Transaction Amount')
plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\dup.png',
            dpi=150, bbox_inches='tight')
plt.show()

Every bar in the top 15 duplicate groups is red  meaning all of the most repeated transaction patterns contain confirmed fraud. The most common duplicate amount is $1.00, appearing repeatedly across different minute buckets. This is a textbook card testing pattern  fraudsters use stolen card details to make tiny $1 charges to verify the card is active before escalating to larger purchases. The concentration of $0.00 transactions also suggests authorisation-only tests with no actual charge.

## Finding 7 — Feature Correlation with Fraud

In [ ]:

df = pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\data\creditcard.csv')

feature_cols = [f'V{i}' for i in range(1, 29)] + ['Amount']
correlations = df[feature_cols + ['Class']].corr()['Class'].drop('Class')


corr_sorted = correlations.abs().sort_values(ascending=False).head(15)


colors = ['#C0392B' if correlations[i] < 0 else '#2E75B6'
          for i in corr_sorted.index]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(corr_sorted.index, corr_sorted.values, color=colors)

ax.set_title('Top 15 Features Correlated with Fraud\n(Red = negative, Blue = positive)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Absolute Correlation with Fraud (Class)')
ax.axvline(0.1, color='grey', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\featureFraud.png',
            dpi=150, bbox_inches='tight')
plt.show()


print('Top 3 fraud signal features:')
print(corr_sorted.head(3))


V17 is the strongest fraud signal with an absolute correlation of 0.33, followed closely by V14 (0.30) and V12 (0.26). All top features except V2, V4, and V11 are negatively correlated (red bars)  meaning fraud transactions have abnormally LOW values for these features compared to legitimate transactions. Features V6, V5, V9, and V1 fall below the 0.10 threshold line, indicating they are weak signals with little predictive value. Critically, Amount is not visible in the top 15 at all  confirming that transaction size alone is a poor fraud indicator and explaining why the flag system uses V17 as its primary feature signal rather than Amount.

 
## Flag System — Rule-Based Fraud Detection

So far the analysis has identified clear patterns in the data:
- Fraud peaks at Hour 2 and Hour 4
- V17 is the strongest fraud signal with 0.33 correlation
- Anomalous transaction amounts are linked to higher fraud rates

The question now is  can we encode these patterns into a simple 
scoring system that automatically identifies the most suspicious 
transactions?

That is exactly what the flag system does.

### How It Works

Every transaction is checked against 3 rules derived directly 
from the analysis above:

**Flag 1 — Anomalous Amount**
If the transaction amount has a Z-score above 3, it sits in the 
top 0.3% of all amounts — statistically abnormal. Directly from 
the amount distribution analysis.

**Flag 2 — Peak Fraud Hour**
If the transaction occurred at Hour 2 or Hour 4 — the two hours 
identified as highest risk in the hourly analysis. Fraud rate at 
these hours is 6-10x the baseline.

**Flag 3 — Abnormal V17**
If V17 falls below the 1st percentile — the bottom 1% of all V17 
values. V17 was identified as the single strongest fraud signal 
with an absolute correlation of 0.33.


### Risk Score

Each flag triggered adds 1 point to the risk score:

| Score | Meaning |
|-------|---------|
| 0 | No flags — almost certainly legitimate |
| 1 | One flag — weak signal, monitor only |
| 2 | Two flags — suspicious, review recommended |
| 3 | All flags — extremely suspicious, urgent review |

### Why No Machine Learning?

In regulated environments like banking, every fraud decision must 
be explainable to auditors and regulators. A rule-based system is 
fully transparent each flag has a clear, data-backed reason.
No black box. Every decision is auditable.

In [ ]:

def build_fraud_flags(df):
  
    flags = pd.DataFrame(index=df.index)

    
    
    z_scores = stats.zscore(df['Amount'])
    flags['high_amount_flag'] = (z_scores > 3).astype(int)

    df2 = df.copy()
    df2['hour'] = (df2['Time'] % 86400 / 3600).astype(int)
    high_risk_hours = [2, 3, 4]  
    flags['odd_hour_flag'] = df2['hour'].isin(high_risk_hours).astype(int)

    
    v14_threshold = df['V14'].quantile(0.01)
    flags['v14_anomaly_flag'] = (df['V14'] < v14_threshold).astype(int)

    
    flags['risk_score']   = flags[['high_amount_flag',
                                   'odd_hour_flag',
                                   'v14_anomaly_flag']].sum(axis=1)
    flags['actual_fraud'] = df['Class'].values
    return flags


flags_df = build_fraud_flags(df)


print('Risk Score Analysis:')
print('-' * 55)
for score in [0, 1, 2, 3]:
    subset    = flags_df[flags_df['risk_score'] == score]
    precision = subset['actual_fraud'].mean() * 100
    count     = len(subset)
    print(f'Score {score}: {count:>7,} transactions | {precision:.2f}% are actual fraud')


In [ ]:
score_data = []
for score in [0, 1, 2, 3]:
    subset    = flags_df[flags_df['risk_score'] == score]
    precision = subset['actual_fraud'].mean() * 100
    score_data.append({
        'risk_score': score,
        'precision':  precision,
        'count':      len(subset)
    })
score_df = pd.DataFrame(score_data)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()   


ax1.bar(score_df['risk_score'], score_df['count'],
        color='#2E75B6', alpha=0.5, label='Transactions Flagged')


ax2.plot(score_df['risk_score'], score_df['precision'],
         color='#C0392B', linewidth=2.5, marker='o', markersize=10,
         label='Actual Fraud %')

ax1.set_xlabel('Risk Score (0=no flags, 3=all flags triggered)')
ax1.set_ylabel('Number of Transactions Flagged', color='#2E75B6')
ax2.set_ylabel('% That Are Actually Fraud', color='#C0392B')
ax1.set_title('Rule-Based Flag System Performance\nHigher score = fewer flags but much higher precision',
              fontsize=13, fontweight='bold')
ax1.set_xticks([0, 1, 2, 3])
plt.tight_layout()
plt.savefig(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\charts\flagscore.png',
            dpi=150, bbox_inches='tight')
plt.show()


The flag system works exactly as expected the more rules a transaction breaks, the more likely it turns out to be fraud.
Transactions with a score of 0 look completely normal. Nothing unusual about the amount, the time, or the V17 value. These make up the bulk of the dataset and their fraud rate sits right at the 0.17% baseline — essentially background noise.
A score of 1 is a weak signal on its own. One suspicious characteristic is not enough to be confident plenty of genuine transactions happen at 2AM or have slightly unusual amounts.
Score 2 is where it gets interesting. When two flags trigger together, the fraud rate jumps to around 40%. That means if a bank investigator had to choose between randomly reviewing transactions and reviewing only Score 2 flagged ones, they would be right 4 out of 10 times instead of less than 1 in 1000. That is a massive improvement.
Score 3 is remarkable. When all three rules fire simultaneously — unusual amount, wrong hour, abnormal V17 nearly every single transaction turns out to be genuine fraud. Instead of searching through 284,807 transactions, you are looking at a tiny handful with near-certainty.
The honest limitation is that this system will miss fraud that does not fit these three patterns. A fraudster who operates at noon with a normal amount would slip through completely. That is the real tradeoff — very high precision, but not full coverage. In practice, Score 3 would trigger an immediate human review rather than an automatic block, which handles the false positive problem cleanly.

---

## Conclusion

Seven patterns were investigated across 284,807 transactions. 
Five produced actionable findings:

**1. Fraud is extremely rare but concentrated**  
Only 0.17% of transactions are fraud. Standard accuracy metrics 
are meaningless — a model predicting "always legitimate" would be 
99.83% accurate but catch nothing.

**2. Fraud peaks at specific hours**  
Hour 2 (1.71% fraud rate) and Hour 4 (1.04% fraud rate) are 
6-10x more dangerous than the baseline. Fraudsters operate when 
monitoring is lowest and transaction volumes are minimal.

**3. Two distinct fraud strategies exist by amount**  
Large transactions ($1000+) show 0.29% fraud — high-value card 
abuse. Micro transactions (under $10) show 0.26% — card testing. 
The same dataset, two completely different criminal behaviours.

**4. Duplicate low-value transactions are a strong fraud signal**  
All top 15 duplicate transaction groups contained confirmed fraud. 
Repeated $1.00 and $0.00 charges in the same minute are textbook 
automated card verification behaviour.

**5. V17 is the strongest fraud indicator**  
Of all 28 anonymised features, V17 has the highest correlation 
with fraud at 0.33. Transaction amount does not appear in the top 
15 — size alone is a poor fraud signal.

---

### Flag System Performance

Encoding these findings into three rules produced a scoring system 
that dramatically outperforms random chance:

| Risk Score | Transactions | Fraud Rate |
|------------|-------------|------------|
| 0 — No flags | 269,083 | 0.02% |
| 1 — One flag | 15,495 | 2.14% |
| 2 — Two flags | 228 | 40.35% |
| 3 — All flags | 1 | 100.00% |

At Score 2, the system is 237x more precise than the 0.17% baseline.  
In production, Score 2 transactions would be the primary threshold 
for human review  reducing the manual workload from 284,807 
transactions to 228 while catching fraud at 40% precision.

---

### Limitations

- Dataset covers only 48 hours — day on day trend analysis is limited
- V1-V28 features are anonymised business interpretation is not possible
- Score 3 flagged only 1 transaction  too small for statistical confidence
- Rule-based systems catch only known patterns  novel fraud methods 
  would be missed

---

*Analysis by Richa | Tools: Python, MySQL, Pandas, Matplotlib, Scipy*